# 01 — Dataset Formation

Download MOEX candles via Tinkoff API → parquet cache.

Core logic: `src/fqw/data/api.py`

In [ ]:
# !pip install -e "..[data]"

import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from fqw.data.api import get_candles_data, INTERVALS
from fqw.data import clean_market_data, fill_time_gaps, prepare_daily_data
from fqw.config import DEFAULT_TICKERS

load_dotenv()
TOKEN = os.getenv("TOKEN")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Intervals:", list(INTERVALS.keys()))

## Download single ticker

In [ ]:
TICKER = "SBER"

df = get_candles_data(
    TICKER,
    TOKEN,
    data_dir=DATA_DIR,
    interval_name="5min",
    start_date=datetime(2020, 1, 1),
    end_date=datetime(2026, 1, 1),
)
df = fill_time_gaps(df)
df = clean_market_data(df)
df.head()

## Batch download thesis tickers

In [ ]:
for ticker in DEFAULT_TICKERS:
    try:
        get_candles_data(ticker, TOKEN, data_dir=DATA_DIR, start_date=datetime(2020, 1, 1))
        print(f"OK {ticker}")
    except Exception as e:
        print(f"FAIL {ticker}: {e}")

## MOEX index (daily)

In [ ]:
moex = get_candles_data("MOEX", TOKEN, data_dir=DATA_DIR, interval_name="1day", start_date=datetime(2015, 1, 1))
moex_daily = prepare_daily_data(moex)
moex_daily.tail()